# Lab 3 — Feature Engineering

**DS331 · Introduction to Data Science**

**Feature engineering** means creating *new columns* from existing ones to make patterns easier to see. The raw data often hides its most interesting stories: the Titanic data has `sibsp` (siblings/spouses aboard) and `parch` (parents/children aboard), but the question "did people traveling *alone* survive less often?" needs a column that doesn't exist yet — until we build it.

In this notebook you will learn how to:

1. [Create features with column arithmetic](#1)
2. [Bin numbers into categories — `pd.cut` and `pd.qcut`](#2)
3. [Extract features from dates and times](#3)
4. [Recode categories — `map` and `replace`](#4)
5. [Custom transformations — `apply` and lambda](#5)
6. [Group-level features — `groupby().transform()`](#6)
7. [One-hot encoding — `pd.get_dummies`](#7)
8. [Try it yourself ✏️](#8)

**Datasets:** *Titanic* (passengers) and *Taxis* (NYC taxi rides — great for datetime features).

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

titanic = sns.load_dataset("titanic")
taxis = sns.load_dataset("taxis")

print("titanic:", titanic.shape, "| taxis:", taxis.shape)

<a id="1"></a>
## 1. Column arithmetic — the simplest features

You can combine columns with ordinary math operators; pandas applies the operation row by row.

**Example 1 — family size.** `sibsp` + `parch` + the passenger themself:

In [ ]:
titanic["family_size"] = titanic["sibsp"] + titanic["parch"] + 1
titanic["family_size"].value_counts().sort_index()

**Example 2 — a boolean flag.** A comparison produces True/False per row — instantly a useful feature:

In [ ]:
titanic["is_alone"] = titanic["family_size"] == 1

# Does the new feature reveal anything? Survival rate, alone vs not:
titanic.groupby("is_alone")["survived"].mean().round(3)

It does: passengers traveling alone survived noticeably less often (~30% vs ~51%). **This finding was invisible before we engineered the feature** — that is the whole point.

**Example 3 — ratios (taxis).** Ratios often say more than raw values: a $30 fare means something different for a 1-mile ride vs a 15-mile ride.

In [ ]:
taxis["fare_per_mile"] = taxis["fare"] / taxis["distance"]

# Dividing by 0 (rides with distance 0) gives inf — replace with NaN
taxis["fare_per_mile"] = taxis["fare_per_mile"].replace(np.inf, np.nan)

taxis["fare_per_mile"].describe().round(2)

> ⚠️ **Common mistake:** division features blow up when the denominator is 0 (`inf`) or missing. Always run `.describe()` on a new feature — an absurd max is a bug, not a discovery.

<a id="2"></a>
## 2. Binning — turning numbers into categories

Continuous values are sometimes easier to analyze (and plot) in groups: ages → age groups, incomes → income brackets.

### 2.1 `pd.cut` — bins at boundaries **you** choose

In [ ]:
titanic["age_group"] = pd.cut(
    titanic["age"],
    bins=[0, 12, 19, 35, 60, 100],
    labels=["child", "teen", "young adult", "adult", "senior"],
)
titanic["age_group"].value_counts()

In [ ]:
# Was "women and children first" real? The engineered feature answers directly:
titanic.groupby("age_group", observed=True)["survived"].mean().round(2)

Children had a ~58% survival rate vs ~27% for seniors. Note the bin edges are *(0, 12], (12, 19], …* — each bin includes its right edge by default.

### 2.2 `pd.qcut` — bins with **equal numbers of rows**

`qcut` chooses the boundaries for you so that each bin holds the same share of the data — perfect for "low / medium / high" style groups:

In [ ]:
titanic["fare_level"] = pd.qcut(titanic["fare"], q=4,
                                labels=["low", "mid-low", "mid-high", "high"])

titanic.groupby("fare_level", observed=True)["survived"].mean().round(2)

A clean gradient: the more you paid, the likelier you survived.

**`cut` vs `qcut`:** use `cut` when the boundaries are meaningful in the real world (legal adult age, price thresholds); use `qcut` when you just want balanced groups.

<a id="3"></a>
## 3. Datetime features

Dates are a goldmine. One timestamp column hides the hour, weekday, month, season… The taxis dataset has real pickup/dropoff timestamps:

In [ ]:
taxis[["pickup", "dropoff"]].head(3)

In [ ]:
# Step 1: make sure they are real datetimes, not text
taxis["pickup"] = pd.to_datetime(taxis["pickup"])
taxis["dropoff"] = pd.to_datetime(taxis["dropoff"])
taxis[["pickup", "dropoff"]].dtypes

### 3.1 The `.dt` accessor

Once a column is a datetime, `.dt` unlocks all its parts:

In [ ]:
taxis["pickup_hour"] = taxis["pickup"].dt.hour
taxis["pickup_day"] = taxis["pickup"].dt.day_name()
taxis["is_weekend"] = taxis["pickup"].dt.dayofweek >= 5    # Mon=0 ... Sun=6

taxis[["pickup", "pickup_hour", "pickup_day", "is_weekend"]].head()

### 3.2 Durations — subtracting datetimes

Subtracting two datetime columns gives a *timedelta*; `.dt.total_seconds()` converts it to a number you can analyze:

In [ ]:
taxis["duration_min"] = (taxis["dropoff"] - taxis["pickup"]).dt.total_seconds() / 60
taxis["duration_min"].describe().round(1)

In [ ]:
# Do the new features reveal patterns? Rides per pickup hour:
taxis["pickup_hour"].value_counts().sort_index().plot(kind="bar", figsize=(10, 3))
plt.title("Number of taxi rides by hour of day")
plt.xlabel("hour")
plt.ylabel("rides")
plt.show()

The rush hours stand out immediately — an engineered feature plus a one-line plot is already a report-worthy figure.

<a id="4"></a>
## 4. Recoding categories — `map` and `replace`

### 4.1 `map` with a dictionary

`map` looks up every value in a dictionary — ideal for translating codes into readable labels (or the reverse):

In [ ]:
titanic["class_label"] = titanic["pclass"].map({1: "First", 2: "Second", 3: "Third"})
titanic[["pclass", "class_label"]].drop_duplicates().sort_values("pclass")

> ⚠️ **Common mistake:** any value *not* in the dictionary becomes `NaN` with `map`. If you only want to change *some* values and keep the rest, use `replace` instead:

In [ ]:
s = pd.Series(["yes", "no", "y", "n", "yes"])
s.replace({"y": "yes", "n": "no"}).value_counts()    # only y/n changed, yes/no untouched

<a id="5"></a>
## 5. Custom transformations — `apply` and lambda

When no built-in function fits, `apply` runs *your own function* on every value. The function is often written inline as a `lambda` (a one-line anonymous function):

In [ ]:
# lambda refresher: these two are equivalent
def double(x):
    return x * 2

double_l = lambda x: x * 2

print(double(21), double_l(21))

In [ ]:
# A feature that needs custom logic: tipping behaviour categories
def tip_category(row):
    if row["fare"] == 0:
        return "unknown"
    pct = row["tip"] / row["fare"] * 100
    if pct == 0:
        return "no tip"
    elif pct < 15:
        return "low"
    elif pct < 25:
        return "standard"
    return "generous"

taxis["tip_level"] = taxis.apply(tip_category, axis=1)   # axis=1 -> row by row
taxis["tip_level"].value_counts()

`axis=1` means the function receives one **row** at a time (so it can use several columns). For a single column, apply directly on it: `df["col"].apply(lambda x: ...)`.

> ⚠️ `apply` is flexible but **slow** on large data. If plain arithmetic (Section 1) or `map` can do the job, prefer those.

<a id="6"></a>
## 6. Group-level features — `groupby().transform()`

Sometimes a row is best understood *relative to its group*: is this ride expensive *for its pickup borough*?

`transform` computes a statistic per group and broadcasts it back onto every row (unlike `.agg()`, which returns one row per group):

In [ ]:
taxis["borough_avg_fare"] = taxis.groupby("pickup_borough")["fare"].transform("mean")
taxis["fare_vs_borough"] = taxis["fare"] / taxis["borough_avg_fare"]

taxis[["pickup_borough", "fare", "borough_avg_fare", "fare_vs_borough"]].head()

Now `fare_vs_borough > 1` means "more expensive than typical for that borough" — a context-aware feature in two lines.

<a id="7"></a>
## 7. One-hot encoding — `pd.get_dummies`

Many statistical and machine-learning methods need *numbers*, not labels. **One-hot encoding** turns one categorical column into several 0/1 columns:

In [ ]:
pd.get_dummies(titanic["embark_town"], prefix="town").head()

In [ ]:
# Usually applied to the whole DataFrame at once:
encoded = pd.get_dummies(titanic[["sex", "embark_town"]], dtype=int)
encoded.head()

For pure EDA you rarely need this — but you *will* need it the moment you fit a model, so it belongs in your toolbox. (Related: `drop_first=True` drops one redundant column per feature.)

### Recap — the feature engineering toolbox

| Tool | Creates | Example |
|---|---|---|
| column arithmetic | numeric features, flags | `family_size`, `is_alone` |
| `pd.cut` / `pd.qcut` | categories from numbers | `age_group`, `fare_level` |
| `.dt` accessor | date/time parts, durations | `pickup_hour`, `duration_min` |
| `map` / `replace` | recoded labels | `class_label` |
| `apply` + lambda | anything custom | `tip_level` |
| `groupby().transform()` | group-relative features | `fare_vs_borough` |
| `pd.get_dummies` | 0/1 columns for models | `town_Cherbourg`, … |

**For your report:** 2–4 *well-motivated* features beat 20 random ones. For each, write one sentence: what it is, and what question it helps answer.

<a id="8"></a>
## 8. Try it yourself ✏️

All exercises use the `taxis` DataFrame (already loaded and with datetimes converted).

**Exercise 1.** Create `total_cost = fare + tip + tolls`, then use `pd.qcut` to bin it into `"cheap"`, `"normal"`, `"expensive"` (3 equal-sized groups). How many rides fall in each?

**Exercise 2.** Create `speed_mph = distance / (duration_min / 60)`. Check it with `.describe()` — do you find suspicious values? Where do they come from?

**Exercise 3.** Using `map`, create a `payment_short` column where `"credit card"` → `"card"` and `"cash"` stays `"cash"`.

**Exercise 4.** Using `groupby().transform()`, add the average `tip` per `pickup_day`, then find which day has the most generous average tip.

**Exercise 5.** Validate one of your new features with a quick plot (e.g. a bar chart of average speed per pickup hour). Write one sentence about what it shows.

In [ ]:
# Exercise 1

In [ ]:
# Exercise 2

In [ ]:
# Exercise 3

In [ ]:
# Exercise 4

In [ ]:
# Exercise 5

---
**Next lab:** with clean data and good features in hand, we get to the heart of the course: **Lab 4 — Data Visualization with Matplotlib and Seaborn**.